In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table("workspace.bronze.orders")

df = (
    df
    .withColumn("order_date", F.to_timestamp("order_date", "dd-MM-yyyy HH:mm"))
    .withColumn("updated_at",F.to_timestamp("updated_at", "dd-MM-yyyy HH:mm"))
    .withColumn("total_amount", F.col("total_amount").cast("decimal(12,2)"))
    .withColumn("order_status", F.upper(F.trim("order_status")))
)
df.show()

In [0]:

window = (
    Window
    .partitionBy("order_id")
    .orderBy(F.col("updated_at").desc())
)

silver_orders = (
    df
    .withColumn("rn", F.row_number().over(window))
    .filter("rn = 1")
    .drop("rn")
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("customer_id").isNotNull())
)

(
    silver_orders.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.orders")
)



In [0]:
display(silver_orders)